# Assignment 7: Neural Network (ANN) Implementation — Shallow vs Deep
**Student ID:** 220119

This notebook implements, tunes, and compares two distinct **Neural Network** architectures:

* **Shallow Neural Network** — exactly one hidden layer
* **Deep Neural Network** — at least three hidden layers with regularization

Both models are built with **PyTorch** and trained on the **Teens Mental Health Dataset**
to predict the binary target `depression_label` (0 = Not Depressed, 1 = Depressed).
The workflow is fully automated so the notebook can be opened and run end-to-end in
**Google Colab** via **Runtime → Run all** without any manual file upload.

## Dataset
* **Source:** Teens Mental Health Dataset (same dataset used in Labs 2, 3, and 4)
* **Target:** `depression_label` (binary: 0 = Not Depressed, 1 = Depressed)
* **Features:** age, gender, daily_social_media_hours, platform_usage, sleep_hours,
  screen_time_before_sleep, academic_performance, physical_activity,
  social_interaction_level, stress_level, anxiety_level, addiction_level

## Pipeline
1. Load data automatically from a public GitHub raw URL (no manual upload).
2. Perform exploratory data analysis (EDA).
3. Preprocess the data: handle missing values, encode categoricals, stratified split, scale features.
4. Build and tune a **Shallow NN** (1 hidden layer) with a validation split.
5. Build and tune a **Deep NN** (3+ hidden layers) with dropout regularization.
6. Generate a 2×1 side-by-side comparison of all required visualisations.
7. Provide a critical analysis contrasting the two architectures.

## 0. Imports and Configuration

In [ ]:
import os
import warnings
import random
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SCREENSHOT_DIR = 'screenshots'
os.makedirs(SCREENSHOT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150})
sns.set_style('whitegrid')
print('Environment ready.')

## 1. Load the Dataset Automatically

The notebook tries a local copy first, then automatically downloads the dataset from
the public GitHub raw link. This enables **Run All** on Colab with no manual uploads.

In [ ]:
# Try local file first, then download from public GitHub raw link.
local_path = os.path.join('dataset', 'Teen_Mental_Health_Dataset.csv')
remote_url = (
    'https://raw.githubusercontent.com/ahsanjust/'
    'Artificial-Intelligence-and-Machine-Learning-Lab/master/'
    'Lab_5/ANN/dataset/Teen_Mental_Health_Dataset.csv'
)

if os.path.exists(local_path):
    df = pd.read_csv(local_path)
    print(f'Loaded from local file: {local_path}')
else:
    df = pd.read_csv(remote_url)
    print(f'Loaded from GitHub: {remote_url}')

print('Shape:', df.shape)
print('Columns:', list(df.columns))

## 2. Exploratory Data Analysis (EDA)

### 2.1 Dataset Overview

In [ ]:
print('First five rows:')
display(df.head())

print('\nDataset info:')
df.info()

print('\nStatistical summary:')
display(df.describe(include='all').T)

### 2.2 Class Distribution

In [ ]:
class_counts = df['depression_label'].value_counts().sort_index()
print('Class distribution:')
print(class_counts)
print(f'\nClass 0 (Not Depressed): {(df["depression_label"]==0).sum()} '
      f'({(df["depression_label"]==0).mean()*100:.2f}%)')
print(f'Class 1 (Depressed):      {(df["depression_label"]==1).sum()} '
      f'({(df["depression_label"]==1).mean()*100:.2f}%)')

plt.figure(figsize=(6, 4))
sns.barplot(x=class_counts.index, y=class_counts.values, palette=['#4C72B0', '#C44E52'])
plt.xticks([0, 1], ['Not Depressed (0)', 'Depressed (1)'])
plt.ylabel('Count')
plt.title('Depression Label Distribution')
for i, v in enumerate(class_counts.values):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Class_Distribution.png'), bbox_inches='tight')
plt.show()

### 2.3 Feature Distributions

In [ ]:
num_features = ['age', 'daily_social_media_hours', 'sleep_hours',
                'screen_time_before_sleep', 'academic_performance',
                'physical_activity', 'stress_level', 'anxiety_level', 'addiction_level']

fig, axes = plt.subplots(3, 3, figsize=(15, 11))
axes = axes.flatten()
for i, feat in enumerate(num_features):
    sns.histplot(df[feat], bins=20, kde=True, ax=axes[i], color='teal')
    axes[i].set_title(feat.replace('_', ' ').title())
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Feature_Distributions.png'), bbox_inches='tight')
plt.show()

### 2.4 Correlation Analysis

In [ ]:
df_corr = df.copy()
for col in ['gender', 'platform_usage', 'social_interaction_level']:
    df_corr[col] = LabelEncoder().fit_transform(df_corr[col])

plt.figure(figsize=(10, 8))
sns.heatmap(df_corr.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Correlation_Heatmap.png'), bbox_inches='tight')
plt.show()

target_corr = df_corr.corr()['depression_label'].drop('depression_label').sort_values(ascending=False)
print('\nTop features correlated with depression_label:')
print(target_corr)

## 3. Data Preprocessing

### 3.1 Missing Value Check

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

### 3.2 Encode Categorical Variables

Neural networks require numeric input. We use **Label Encoding** for the three
categorical columns (`gender`, `platform_usage`, `social_interaction_level`):

* `gender` is binary (male/female).
* `platform_usage` and `social_interaction_level` have a natural ordinal structure.

This matches the preprocessing done in all previous labs.

In [ ]:
df_encoded = df.copy()
label_encoders = {}
for col in ['gender', 'platform_usage', 'social_interaction_level']:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

print('Encoding maps:')
for col, le in label_encoders.items():
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

### 3.3 Train / Validation / Test Split

A **stratified** split preserves the severe class imbalance (~2.5% positive) in all
subsets. We use 70% train, 15% validation (for hyperparameter tuning), 15% test.

In [ ]:
X = df_encoded.drop('depression_label', axis=1)
y = df_encoded['depression_label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp)

print(f'Train:      {X_train.shape[0]} samples, {y_train.sum()} positives')
print(f'Validation: {X_val.shape[0]} samples, {y_val.sum()} positives')
print(f'Test:       {X_test.shape[0]} samples, {y_test.sum()} positives')

### 3.4 Feature Scaling

Standardization is essential for neural networks because:
* Features with larger magnitudes would dominate the gradient updates.
* Activation functions (ReLU, Sigmoid) perform best with inputs near zero.
* Gradient-based optimization converges much faster on normalized features.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print('Feature scaling completed!')
print(f'X_train_scaled mean: {X_train_scaled.mean():.6f}')
print(f'X_train_scaled std:  {X_train_scaled.std():.6f}')

n_features = X_train_scaled.shape[1]
print(f'Number of features: {n_features}')

### 3.5 Convert to PyTorch Tensors

In [ ]:
def to_tensor(X, y):
    return (
        torch.tensor(X, dtype=torch.float32).to(DEVICE),
        torch.tensor(y.values, dtype=torch.float32).view(-1, 1).to(DEVICE)
    )

X_train_t, y_train_t = to_tensor(X_train_scaled, y_train)
X_val_t, y_val_t     = to_tensor(X_val_scaled, y_val)
X_test_t, y_test_t   = to_tensor(X_test_scaled, y_test)

print('PyTorch tensors ready.')
print(f'X_train_t shape: {X_train_t.shape}')
print(f'y_train_t shape: {y_train_t.shape}')

## 4. Utility Functions for Training and Evaluation

In [ ]:
class WeightedBCEWithLogitsLoss(nn.Module):
    """Binary cross-entropy with per-class weights to handle imbalance."""
    def __init__(self, pos_weight=1.0):
        super().__init__()
        self.pos_weight = torch.tensor([pos_weight], dtype=torch.float32)

    def forward(self, logits, targets):
        loss = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=self.pos_weight.to(logits.device)
        )
        return loss


def compute_class_weight(y):
    """Compute positive-class weight as n_negative / n_positive."""
    n_neg = (y == 0).sum()
    n_pos = (y == 1).sum()
    return n_neg / n_pos if n_pos > 0 else 1.0


def train_model(model, criterion, optimizer, X_train, y_train, X_val, y_val,
                epochs=100, batch_size=32, verbose=True):
    """
    Train a PyTorch model and return training history.
    Returns dict with train_loss, val_loss, train_acc, val_acc per epoch.
    """
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(1, epochs + 1):
        # --- Training ---
        model.train()
        epoch_train_loss = 0.0
        train_correct = 0
        train_total = 0

        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item() * X_batch.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)

        avg_train_loss = epoch_train_loss / len(dataset)
        train_acc = train_correct / train_total

        # --- Validation ---
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val)
            val_loss = criterion(val_logits, y_val).item()
            val_preds = (torch.sigmoid(val_logits) >= 0.5).float()
            val_acc = (val_preds == y_val).float().mean().item()

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if verbose and epoch % 25 == 0:
            print(f'  Epoch {epoch:3d}/{epochs}  |  Train Loss: {avg_train_loss:.4f}  '
                  f'Val Loss: {val_loss:.4f}  |  Train Acc: {train_acc:.4f}  Val Acc: {val_acc:.4f}')

    return history


def evaluate_model(model, X, y, threshold=0.5):
    """Compute standard classification metrics."""
    model.eval()
    with torch.no_grad():
        logits = model(X)
        y_prob = torch.sigmoid(logits).cpu().numpy().flatten()
    y_true = y.cpu().numpy().flatten()
    y_pred = (y_prob >= threshold).astype(int)

    return {
        'y_true': y_true,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'acc': accuracy_score(y_true, y_pred),
        'prec': precision_score(y_true, y_pred, zero_division=0),
        'rec': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'auc': roc_auc_score(y_true, y_prob),
    }

## 5. Shallow Neural Network Implementation

The shallow network has **exactly one hidden layer**. We hyperparameter-tune:
* Number of hidden units (neurons in the hidden layer)
* Activation function (ReLU vs Sigmoid)
* Batch size

The model uses a validation split to guide tuning.

### 5.1 Define the Shallow NN Architecture

In [ ]:
class ShallowNN(nn.Module):
    """Feedforward neural network with exactly one hidden layer."""
    def __init__(self, input_dim, hidden_dim=64, activation='relu'):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

        if activation == 'relu':
            self.act = nn.ReLU()
        elif activation == 'sigmoid':
            self.act = nn.Sigmoid()
        elif activation == 'tanh':
            self.act = nn.Tanh()
        else:
            self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.fc2(x)   # raw logits — BCEWithLogitsLoss handles sigmoid internally
        return x

    def __repr__(self):
        return (f'ShallowNN(input_dim={self.fc1.in_features}, '
                f'hidden_dim={self.fc1.out_features}, '
                f'act={self.act.__class__.__name__})')

### 5.2 Hyperparameter Tuning — Shallow NN

We tune hidden units, activation function, and batch size using validation set performance.

In [ ]:
shallow_configs = []

hidden_units_list = [16, 32, 64, 128]
activations_list = ['relu', 'sigmoid', 'tanh']
batch_sizes = [16, 32, 64]
shallow_epochs = 100
shallow_lr = 0.001

pos_weight = compute_class_weight(y_train)
print(f'Positive class weight: {pos_weight:.4f}')
print(f'\nTuning Shallow NN over {len(hidden_units_list) * len(activations_list) * len(batch_sizes)} combinations...\n')

best_shallow_val_f1 = 0.0
best_shallow_config = None
best_shallow_model = None
best_shallow_history = None

for hidden_dim in hidden_units_list:
    for activation in activations_list:
        for batch_size in batch_sizes:
            model = ShallowNN(n_features, hidden_dim, activation).to(DEVICE)
            criterion = WeightedBCEWithLogitsLoss(pos_weight)
            optimizer = optim.Adam(model.parameters(), lr=shallow_lr)

            history = train_model(
                model, criterion, optimizer,
                X_train_t, y_train_t, X_val_t, y_val_t,
                epochs=shallow_epochs, batch_size=batch_size, verbose=False
            )

            # Evaluate on validation set
            metrics = evaluate_model(model, X_val_t, y_val_t)
            val_f1 = metrics['f1']

            config = {
                'hidden_dim': hidden_dim,
                'activation': activation,
                'batch_size': batch_size,
            }
            shallow_configs.append({**config, 'val_f1': val_f1, 'val_auc': metrics['auc']})

            if val_f1 > best_shallow_val_f1:
                best_shallow_val_f1 = val_f1
                best_shallow_config = config
                best_shallow_model = model
                best_shallow_history = history

            print(f'  hidden={hidden_dim:3d}  act={activation:7s}  batch={batch_size:2d}  '
                  f'-> Val F1={val_f1:.4f}  Val AUC={metrics["auc"]:.4f}')

shallow_results = pd.DataFrame(shallow_configs)
print(f'\n=== Best Shallow NN Configuration ===')
print(f'  Hidden units: {best_shallow_config["hidden_dim"]}')
print(f'  Activation:   {best_shallow_config["activation"]}')
print(f'  Batch size:   {best_shallow_config["batch_size"]}')
print(f'  Val F1:       {best_shallow_val_f1:.4f}')

### 5.3 Train Final Shallow NN (full train set)

In [ ]:
# Re-train with best config on combined train+val for final model
X_combined_t = torch.cat([X_train_t, X_val_t], dim=0)
y_combined_t = torch.cat([y_train_t, y_val_t], dim=0)

final_shallow = ShallowNN(
    n_features,
    best_shallow_config['hidden_dim'],
    best_shallow_config['activation']
).to(DEVICE)

criterion_shallow = WeightedBCEWithLogitsLoss(pos_weight)
optimizer_shallow = optim.Adam(final_shallow.parameters(), lr=shallow_lr)

print('Training final Shallow NN on combined train+val...')
shallow_history_final = train_model(
    final_shallow, criterion_shallow, optimizer_shallow,
    X_combined_t, y_combined_t, X_test_t, y_test_t,
    epochs=shallow_epochs, batch_size=best_shallow_config['batch_size'], verbose=True
)
print('Shallow NN training complete.')

## 6. Deep Neural Network Implementation

The deep network has **at least three hidden layers** with **Dropout regularization**
to combat overfitting. We hyperparameter-tune:
* Learning rate (0.01 vs 0.001 vs 0.0001)
* Optimizer (Adam vs SGD)
* Number of training epochs

### 6.1 Define the Deep NN Architecture

In [ ]:
class DeepNN(nn.Module):
    """Deep feedforward network with 3 hidden layers, dropout, and layer norm."""
    def __init__(self, input_dim, hidden_dims=[64, 32, 16], dropout_rate=0.3):
        super().__init__()
        layers = []
        prev_dim = input_dim

        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
            ])
            prev_dim = h_dim

        layers.append(nn.Linear(prev_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    def __repr__(self):
        return (f'DeepNN(input_dim={self.network[0].in_features}, '
                f'#layers={(len(self.network)-1)//4})')

### 6.2 Hyperparameter Tuning — Deep NN

We tune learning rate and optimizer (Adam vs SGD) with a fixed architecture.

In [ ]:
deep_configs = []

learning_rates = [0.01, 0.001, 0.0001]
optimizers_to_try = ['adam', 'sgd']
deep_epochs = 150
deep_batch_size = 32
deep_hidden_dims = [64, 32, 16]  # 3 hidden layers
dropout_rate = 0.3

print(f'Tuning Deep NN over {len(learning_rates) * len(optimizers_to_try)} combinations...\n')

best_deep_val_f1 = 0.0
best_deep_config = None
best_deep_model = None
best_deep_history = None

for lr in learning_rates:
    for opt_name in optimizers_to_try:
        model = DeepNN(n_features, deep_hidden_dims, dropout_rate).to(DEVICE)
        criterion = WeightedBCEWithLogitsLoss(pos_weight)

        if opt_name == 'adam':
            optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
        else:
            optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)

        history = train_model(
            model, criterion, optimizer,
            X_train_t, y_train_t, X_val_t, y_val_t,
            epochs=deep_epochs, batch_size=deep_batch_size, verbose=False
        )

        metrics = evaluate_model(model, X_val_t, y_val_t)
        val_f1 = metrics['f1']

        config = {'lr': lr, 'optimizer': opt_name}
        deep_configs.append({**config, 'val_f1': val_f1, 'val_auc': metrics['auc']})

        if val_f1 > best_deep_val_f1:
            best_deep_val_f1 = val_f1
            best_deep_config = config
            best_deep_model = model
            best_deep_history = history

        print(f'  lr={lr:.4f}  opt={opt_name:5s}  -> Val F1={val_f1:.4f}  Val AUC={metrics["auc"]:.4f}')

deep_results = pd.DataFrame(deep_configs)
print(f'\n=== Best Deep NN Configuration ===')
print(f'  Learning rate: {best_deep_config["lr"]}')
print(f'  Optimizer:     {best_deep_config["optimizer"]}')
print(f'  Val F1:        {best_deep_val_f1:.4f}')

### 6.3 Train Final Deep NN (full train set)

In [ ]:
final_deep = DeepNN(n_features, deep_hidden_dims, dropout_rate).to(DEVICE)

criterion_deep = WeightedBCEWithLogitsLoss(pos_weight)
if best_deep_config['optimizer'] == 'adam':
    optimizer_deep = optim.Adam(final_deep.parameters(), lr=best_deep_config['lr'], weight_decay=1e-4)
else:
    optimizer_deep = optim.SGD(final_deep.parameters(), lr=best_deep_config['lr'], momentum=0.9, weight_decay=1e-4)

print('Training final Deep NN on combined train+val...')
deep_history_final = train_model(
    final_deep, criterion_deep, optimizer_deep,
    X_combined_t, y_combined_t, X_test_t, y_test_t,
    epochs=deep_epochs, batch_size=deep_batch_size, verbose=True
)
print('Deep NN training complete.')

## 7. Test-Set Evaluation

In [ ]:
shallow_metrics = evaluate_model(final_shallow, X_test_t, y_test_t)
deep_metrics = evaluate_model(final_deep, X_test_t, y_test_t)

def print_metrics(metrics, name):
    print(f'\n=== {name} ===')
    print(f'  Accuracy:  {metrics["acc"]:.4f}')
    print(f'  Precision: {metrics["prec"]:.4f}')
    print(f'  Recall:    {metrics["rec"]:.4f}')
    print(f'  F1-Score:  {metrics["f1"]:.4f}')
    print(f'  AUC-ROC:   {metrics["auc"]:.4f}')

print_metrics(shallow_metrics, 'Shallow NN')
print_metrics(deep_metrics, 'Deep NN')

## 8. Required Visualisations — 2×1 Comparison Matrix

All visualisations are rendered as side-by-side comparisons (Shallow vs Deep) following
the assignment specification.

### 8.1 Training History — Loss and Accuracy Curves

Line graphs showing training vs validation loss and accuracy across all epochs for both models,
arranged in a 2×1 matrix.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Row 1: Loss Curves ---
for ax, history, name, color in zip(
    axes[0],
    [shallow_history_final, deep_history_final],
    ['Shallow NN', 'Deep NN'],
    ['#4C72B0', '#C44E52']
):
    epochs_range = range(1, len(history['train_loss']) + 1)
    ax.plot(epochs_range, history['train_loss'], color=color, lw=1.5, label='Train Loss')
    ax.plot(epochs_range, history['val_loss'], color=color, lw=1.5, ls='--', label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(f'{name} — Loss Curve')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# --- Row 2: Accuracy Curves ---
for ax, history, name, color in zip(
    axes[1],
    [shallow_history_final, deep_history_final],
    ['Shallow NN', 'Deep NN'],
    ['#4C72B0', '#C44E52']
):
    epochs_range = range(1, len(history['train_acc']) + 1)
    ax.plot(epochs_range, history['train_acc'], color=color, lw=1.5, label='Train Acc')
    ax.plot(epochs_range, history['val_acc'], color=color, lw=1.5, ls='--', label='Val Acc')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'{name} — Accuracy Curve')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training History — Shallow vs Deep NN', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Training_History.png'), bbox_inches='tight')
plt.show()

### 8.2 Confusion Matrix (2×1 Heatmap)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metrics, name in zip(
    axes, [shallow_metrics, deep_metrics], ['Shallow NN', 'Deep NN']
):
    cm = confusion_matrix(metrics['y_true'], metrics['y_pred'])
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
        xticklabels=['Not Depressed (0)', 'Depressed (1)'],
        yticklabels=['Not Depressed (0)', 'Depressed (1)'],
        annot_kws={'size': 13}
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'Confusion Matrix — {name}')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Confusion_Matrix.png'), bbox_inches='tight')
plt.show()

### 8.3 ROC Curve (2×1) with AUC Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, metrics, name in zip(
    axes, [shallow_metrics, deep_metrics], ['Shallow NN', 'Deep NN']
):
    fpr, tpr, _ = roc_curve(metrics['y_true'], metrics['y_prob'])
    ax.plot(fpr, tpr, 'b-', linewidth=2, label=f"{name} (AUC = {metrics['auc']:.4f})")
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')
    ax.fill_between(fpr, tpr, alpha=0.2)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve — {name}')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'ROC_Curve.png'), bbox_inches='tight')
plt.show()

### 8.4 Evaluation Metrics — Combined Grouped Bar Chart

A single grouped bar chart comparing both models across Accuracy, Precision, Recall,
F1-Score, and AUC.

In [ ]:
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC'],
    'Shallow NN': [shallow_metrics['acc'], shallow_metrics['prec'],
                   shallow_metrics['rec'], shallow_metrics['f1'], shallow_metrics['auc']],
    'Deep NN': [deep_metrics['acc'], deep_metrics['prec'],
                deep_metrics['rec'], deep_metrics['f1'], deep_metrics['auc']]
}).melt(id_vars='Metric', var_name='Model', value_name='Score')

plt.figure(figsize=(10, 6))
sns.barplot(data=metrics_df, x='Metric', y='Score', hue='Model', palette=['#4C72B0', '#C44E52'])
plt.ylim(0, 1.05)
plt.ylabel('Score')
plt.title('Shallow NN vs Deep NN — Evaluation Metrics on the Test Set')
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt='%.3f', fontsize=9, padding=2)
plt.legend(title='Model')
plt.tight_layout()
plt.savefig(os.path.join(SCREENSHOT_DIR, 'Metrics_Comparison.png'), bbox_inches='tight')
plt.show()

### 8.5 Network Structure — Sequential Layer Summary

A clean text-based table showing the architectural topology of both models.

In [ ]:
def print_model_summary(model, name):
    print(f'\n{"="*60}')
    print(f'  {name}')
    print(f'{"="*60}')
    total_params = 0
    print(f'  {"Layer":<25} {"Output Shape":<20} {"Params":<10}')
    print(f'  {"-"*55}')
    for i, (name_, module) in enumerate(model.named_modules()):
        if isinstance(module, (nn.Linear, nn.BatchNorm1d, nn.Dropout)):
            if isinstance(module, nn.Linear):
                in_f = module.in_features
                out_f = module.out_features
                n_params = in_f * out_f + out_f
                output_shape = f'(-1, {out_f})'
            elif isinstance(module, nn.BatchNorm1d):
                n_params = 2 * module.num_features
                output_shape = f'(-1, {module.num_features})'
            else:
                n_params = 0
                output_shape = '(-1, *)'
            total_params += n_params
            print(f'  {name_:<25} {output_shape:<20} {n_params:<10}')
    print(f'  {"-"*55}')
    print(f'  {"Total params":<25} {"":<20} {total_params:<10}')
    print(f'{"="*60}\n')

print_model_summary(final_shallow, 'Shallow NN Architecture')
print_model_summary(final_deep, 'Deep NN Architecture')

## 9. Performance Interpretation & Analysis

The table below summarises the test-set metrics for both the optimised Shallow NN
(one hidden layer) and Deep NN (three hidden layers with dropout).

| Metric      | Shallow NN | Deep NN |
|-------------|-----------|---------|
| Accuracy    | see the Test-Set Evaluation cell above |
| Precision   | see the Test-Set Evaluation cell above |
| Recall      | see the Test-Set Evaluation cell above |
| F1-Score    | see the Test-Set Evaluation cell above |
| AUC-ROC     | see the Test-Set Evaluation cell above |

### Critical Analysis

**Contrasting the two architectures:**

* The **Shallow NN** (1 hidden layer) is simpler and faster to train. With the tuned
  configuration printed in the hyperparameter-tuning cell above, it provides a strong baseline.
  The single hidden layer limits the model's capacity to learn complex non-linear
  interactions, but on this dataset the dominant signal comes from a small subset of
  features (`anxiety_level`, `stress_level`), so the shallow model performs competitively.

* The **Deep NN** (3 hidden layers with Batch Normalization and Dropout regularization) has substantially
  more representational capacity. The deeper architecture can model more intricate
  decision boundaries. However, on a small dataset (only 1 200 samples and ~2.5% positive),
  the added capacity creates a risk of **overfitting** — especially if the validation
  curves show divergence from the training curves.

* Both architectures use `class_weight='balanced'` (via the weighted BCE loss) to
  address the severe class imbalance, which is critical for obtaining meaningful
  recall on the minority class.

* **Training History Diagnosis:**
  - If both training and validation loss decrease together and converge, the model
    is learning effectively without overfitting.
  - If training loss continues to decrease while validation loss stagnates or increases,
    this signals overfitting — the deep model may be memorising the training set.
  - The deep NN's dropout and L2 regularization (`weight_decay`) are designed to
    mitigate this, at the cost of slower convergence.

* **Interpretation of the Results:**
  - Examine the F1-score and AUC-ROC — these are the most informative metrics given
    the 50:1 class imbalance.
  - The Confusion Matrix reveals how many minority-class samples each model captures
    (recall for class 1).
  - If the deep NN achieves a **justifiable lift** in F1/AUC over the shallow NN, the
    deeper architecture provides tangible benefit. If the metrics are comparable but
    the deep NN shows signs of overfitting, the simpler shallow model is preferable.

## 10. Classification Report (Test Set)

In [ ]:
print('Shallow NN:')
print(classification_report(shallow_metrics['y_true'], shallow_metrics['y_pred'],
                            target_names=['Not Depressed', 'Depressed'],
                            zero_division=0))

print('\nDeep NN:')
print(classification_report(deep_metrics['y_true'], deep_metrics['y_pred'],
                            target_names=['Not Depressed', 'Depressed'],
                            zero_division=0))

## 11. Summary and Conclusion

### Key Observations

* The **Teens Mental Health Dataset** is severely imbalanced (~2.5% positive class),
  making this a challenging binary classification problem. The 70/15/15 stratified
  split ensures all subsets preserve this ratio.
* Both the Shallow NN (1 hidden layer) and Deep NN (3 hidden layers) use weighted
  binary cross-entropy with class weights computed from the training set to mitigate
  the imbalance.
* The shallow model was tuned over 36 hyperparameter combinations (4 hidden dims ×
  3 activations × 3 batch sizes), while the deep model was tuned over 6 combinations
  (3 learning rates × 2 optimizers).
* The deep NN incorporates Dropout, Batch Normalization, and
  L2 weight decay as regularization strategies against overfitting.

### Comparison Summary

| Aspect          | Shallow NN                     | Deep NN                          |
|-----------------|--------------------------------|----------------------------------|
| Hidden layers   | 1                              | 3                                |
| Regularization  | None                           | Dropout + BatchNorm + L2         |
| Training speed  | Faster                         | Slower (more parameters)         |
| Overfitting risk| Lower                           | Higher (mitigated by regularizers)|

### How to Reproduce

1. Open the notebook in Google Colab.
2. Click **Runtime → Run all**.
3. The dataset is fetched automatically from the public GitHub raw link, both NNs
   are tuned with validation splits, and all five required visualisations (training
   history, confusion matrix, ROC curve, metrics bar chart, network structure) are
   produced without any manual file uploads.

---
*Notebook by Ahsanul Haque (220119)*